# This notebook shows you how to access the remotely stored processed data

# Initial set up

## These steps help you build the project on a local machine. 
If you are using nrp Jupyterhub, it is recommended you use the provided block instead.

#### Build the project
- ```pip install .[plotting]```

#### Install aws cli (optional)
You will want to install this if you want to manually see the data stored in the nrp bucket.

Example for installing on linux :
- ```sudo curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"```
- ```sudo unzip awscliv2.zip```
- ```sudo ./aws/install```

### Note about optional dependencies
If you have already built the project with ```pip install .``` make sure you rebuild with ```pip install .[plotting]``` as we will be using these optional packages here.

## Running in NRP Jupyterhub

This is for running this notebook on nrp Jupyterhub. 

This block simply builds the project and installs dependencies not already present on nrp jupyterhub.
You can safely ignore pip warnings

For serious projects, users should use conda or docker but this notebook is meant to be very simple and user friendly.

In [ ]:
RUNNING_ON_NRP = False
if (RUNNING_ON_NRP):
    %pip install -e ../. --no-deps

    %pip install xgcm
    %pip install zarr
    %pip install boto3
    %pip install ujson
    %pip install scikit-fmm
    %pip install aiobotocore
    %pip install cmocean

## Set AWS credentials
If you are accessing or writing data to S3, you must set credentials.
NOTE : S3 is all that is supported currently.
This typically corresponds to an NRP S3 bucket.

windows:

- ```$env:AWS_ACCESS_KEY_ID="..."```
- ```$env:AWS_SECRET_ACCESS_KEY="..."```

unix:

- ```export AWS_ACCESS_KEY_ID=...```
- ```export AWS_SECRET_ACCESS_KEY=...```

In [ ]:
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
from matplotlib.colors import BoundaryNorm

import dbof.cutout_dataset_creation.zarr_dataset as zarr_dataset
import dbof.io.filesystems as filesystems

# Load DBOF

## Dataset access config

For convenience the parameters used in accessing the dataset can be stored in config files under ```configs/cutouts/data_access/```.
These values must match the run configurations.

Example config:
```
data_access:
  s3_endpoint: "https://s3-west.nrp-nautilus.io"
  bucket: "llc"
  folder: "native_grid_dbof_training_data"
  run_id: "year_4x150"
```

If the user prefers, these values can also simply be hardcoded when using the package.

In [ ]:
#path_to_config = "../../configs/cutouts/data_access/1year_4_snapshots.yaml"
path_to_config = "../../configs/cutouts/data_access/test.yaml"
with open(path_to_config) as f:
    data_cfg = yaml.safe_load(f)["data_access"]

bucket = data_cfg["bucket"]
folder = data_cfg["folder"]
s3_endpoint = data_cfg["s3_endpoint"]
feature_channels = data_cfg["feature_channels"]
run_id = data_cfg["run_id"] # If you generated your own dataset make sure and update the run_id here or in your own config.

run_id = "big_run_00"


# User can also simply do
# bucket = "bucket"
# folder = "folder"
# .
# .
# .

In [ ]:
# Build our file system for accessing remote files
fs, fs_synch = filesystems.create_s3_filesystems(s3_endpoint)

# Metadata
First we will load in the metadata. The metadata has per cutout information for each datapoint in our dataset

In [ ]:
# A dataset run generates many metadata files.
files = fs_synch.glob(
    f"{bucket}/{folder}/{run_id}/metadata/*.parquet"
)

meta_df = pd.read_parquet(files, filesystem=fs_synch)

In [ ]:
# A quick look at the metadata
meta_df

### A pdf of the log divergence of buoyancy squared found in our metadata
This value is stored in the metadata as a center point as this center point is what the data is currently sampled by.

In [ ]:
hist, edges = np.histogram(meta_df["log_grad_b_2_center"], bins=10, density=True)

plt.bar(
    edges[:-1],
    hist,
    width=np.diff(edges),
    align="edge"
)

plt.xlabel("Log(B)")
plt.ylabel("Density")
plt.title("PDF of log(B) \nBias = 1.3")

plt.show()

### A look at the geospatial sampling location of each cutout

In [ ]:
lats = meta_df["center_lat"].values
lons = meta_df["center_lon"].values

timestamps = pd.to_datetime(meta_df["time_snapshot"])

timestamps = pd.Series(u.strftime("%Y-%m-%d") for u in timestamps)

unique_times = timestamps.sort_values().unique()

time_to_index = {t: i for i, t in enumerate(unique_times)}

time_indices = np.array([time_to_index[t] for t in timestamps])
n_times = len(unique_times)

# discrete colormap
cmap = plt.get_cmap("tab10")#, n_times)
norm = BoundaryNorm(np.arange(-0.5, n_times + 0.5, 1), n_times)


fig = plt.figure(figsize=(11, 5))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.set_global()
ax.coastlines()

sc = ax.scatter(
    lons,
    lats,
    c=time_indices,
    cmap=cmap,
    norm=norm,
    s=8,
    alpha=0.8,
    transform=ccrs.PlateCarree()
)

# discrete colorbar with timestamp labels
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation="horizontal",
    pad=0.06,
    ticks=np.arange(n_times)
)

cbar.set_ticklabels([str(t) for t in unique_times])
cbar.set_label("Time Snapshot")

plt.title("Sampled Points By Year \nBias = 1.3")
plt.show()

In [ ]:
lats = meta_df["center_lat"].values
lons = meta_df["center_lon"].values


cmap = plt.get_cmap("tab10")

fig = plt.figure(figsize=(11, 5))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.set_global()
ax.coastlines()

sc = ax.scatter(
    lons,
    lats,
    cmap=cmap,
    s=8,
    alpha=0.05,
    transform=ccrs.PlateCarree()
)

plt.title("Sampled Points By Year \nBias = 1.3")
plt.show()

# Load the data cutouts

In [ ]:
# This reader class will allow easy access to the remotely stored data

reader = zarr_dataset.ZarrDatasetReader(
    bucket=bucket,
    folder=folder,
    run_id=run_id,
    dataset_name="cutout_dataset_creation.zarr",
    fs=fs
)

### Plot some data samples and cooresponding metadata

In [ ]:
# todo it would be cool to add a lat lon pin to each sample

import cmocean


def plot_image_features(imgs, ids):
    """
    imgs: (N, C, H, W)
    """

    N, C, H, W = imgs.shape

    fig = plt.figure(figsize=(2.5*(C+1), 2.5*N))
    gs = fig.add_gridspec(N, C + 1, wspace=0.5, hspace=0.15)

    for i in range(N):
        row_data = imgs[i]

        for c in range(C):
            ax = fig.add_subplot(gs[i, c])

            # vmin = np.nanmin(row_data[c])
            # vmax = np.nanmax(row_data[c])


            if c == 0:
                cmap = cmocean.cm.gray
            elif c == 1 :
                cmap = cmocean.cm.haline
            elif c == 2 :
                cmap = cmocean.cm.thermal
            elif c == 6 :
                cmap = cmocean.cm.diff
            else :
                cmap = cmocean.cm.speed



            im = ax.imshow(row_data[c], cmap=cmap) #, vmin=vmin, vmax=vmax)
            ax.set_xticks([])
            ax.set_yticks([])

            if i == 0:
                ax.set_title(f"{feature_channels[c]}", fontsize=10)

            fig.colorbar(
                im,
                ax=ax,
                fraction=0.046,
                pad=0.01 #format="%4.2"
            )

    plt.show()

rand_indices = np.random.randint(0, reader.num_images, size=4)
imgs, ids = reader.get_images(rand_indices)

plot_image_features(imgs, ids)

# Metadata for each sample
subset_df = (
    meta_df
    .set_index("dataset_index")
    .loc[ids]
    .reset_index()
)
subset_df

In [ ]:
# Plot out every feature of an image from our data

img, img_id = reader.get_image(10)
for i in range(len(img)):

    plt.figure(figsize=(8,8), dpi= 90)
    plt.imshow(img[i],origin='lower',cmap='jet')

# Load full data into ram for training
The reader provides a dask array accessor to access the entire dataset lazily.
Here is a recommended approach to loading in data.

In [ ]:
# get the dask arrays for images, ids and a validity mask that masks out empty ids
images_da, ids_da, valid_mask_da = reader.full_dataset_as_dask()

In [ ]:
# option 1 load the masks into memory and leave the images and ids lazy then compute
from dask import array as da

valid_idx = da.nonzero(valid_mask_da)[0].compute()   # numpy array in RAM
valid_images = da.take(images_da, valid_idx, axis=0).compute()
valid_ids = da.take(ids_da, valid_idx, axis=0).compute()

In [ ]:
# option 2 load ids and images into memory and compute masks locally
# this is the fastest option but requires the most ram
images = images_da.compute()
ids = ids_da.compute()

mask = (ids != b"")  # or NUL-safe logic if needed
images = images[mask]
ids = ids[mask]

In [ ]:
# plot
rand_indices = np.random.randint(0, len(valid_images), size=4)
imgs = valid_images[rand_indices]
ids = valid_ids[rand_indices]

plot_image_features(imgs, ids)

# Metadata for each sample
subset_df = (
    meta_df
    .set_index("dataset_index")
    .loc[ids]
    .reset_index()
)
subset_df